# Notebook 1 — Exploring the ADMET Scorer & Defining Tools

**Goal:** Understand what the local RDKit-based ADMET scorer returns, then formalise it as a tool the Claude agent can call.

### What we cover
1. Score a real molecule (Aspirin) using the local RDKit scorer
2. Understand every field in the response
3. Extract the key scores the agent will reason about
4. Define the three **Claude tool schemas** the agent will use

---
> **Scoring:** All property calculations run locally via `agent_utils.py` using RDKit.  
> **No network required** — instant results, no API key needed for the scoring step.

In [ ]:
# Install dependencies (run once)
# !pip install requests pandas rdkit anthropic
# If rdkit fails: conda install -c conda-forge rdkit

In [ ]:
import sys, os, json, time
import pandas as pd

# Add parent directory so we can import agent_utils
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from agent_utils import call_admet_api, extract_key_scores, is_valid_smiles

print('Imports OK')

---
## 1. Score a Molecule — Raw Response

We send **Aspirin** (`CC(=O)OC1=CC=CC=C1C(=O)O`) through the local RDKit scorer and look at everything that comes back.

In [ ]:
ASPIRIN = 'CC(=O)OC1=CC=CC=C1C(=O)O'

t0 = time.time()
raw = call_admet_api(ASPIRIN)
print(f'Scored in {time.time()-t0:.3f}s')

# Pretty-print the full response
print(json.dumps(raw, indent=2, default=str))

---
## 2. Response Structure

The local scorer returns a rich JSON with these top-level keys:

| Field | What it is |
|---|---|
| `canonical_smiles` | RDKit-canonicalized SMILES |
| `molecular_weight`, `clogp`, `tpsa`, `hbd`, `hba` | Physicochemical descriptors |
| `qed` | `{qed_score, qed_class}` — overall drug-likeness (0-1) |
| `lipinski_summary` | `Pass / Borderline / Fail` |
| `admet` | Nested: `solubility`, `gi_absorption`, `bbb_penetration`, `cns_mpo`, `cyp_metabolism`, `toxicity` |
| `alerts` | List of PAINS / structural alerts `[{name, description}]` |
| `fail_fast_score` | Composite triage score (0-10) |
| `decision` | `Progress / Optimize / Kill` — triage verdict |
| `decision_rationale` | Human-readable explanation of the decision |
| `optimization_suggestions` | `[{text}]` — rule-based improvement hints |

The agent will use all of these to reason about what to change.

In [ ]:
# Extract the fields the agent will reason about
scores = extract_key_scores(raw)

# Display as a readable table
pd.Series(scores).to_frame('value')

---
## 3. Compare a Few Reference Molecules

Before building the agent, let's build intuition by comparing molecules we know well.

In [ ]:
MOLECULES = {
    'Aspirin':      'CC(=O)OC1=CC=CC=C1C(=O)O',
    '(S)-Ibuprofen':'CC(C)Cc1ccc([C@H](C)C(=O)O)cc1',
    'Atenolol':     'CC(C)NCC(O)COc1ccc(CC(N)=O)cc1',  # BBB− by design (CNS demo start)
    'Propranolol':  'CC(C)NCC(O)COc1cccc2ccccc12',      # older beta-blocker, BBB+ (contrast)
}

rows = []
for name, smiles in MOLECULES.items():
    print(f'Analyzing {name}...')
    resp = call_admet_api(smiles)
    if 'error' not in resp:
        s = extract_key_scores(resp)
        rows.append({
            'Molecule':       name,
            'MW':             s['molecular_weight'],
            'cLogP':          s['clogp'],
            'QED':            s['qed_score'],
            'Lipinski':       s['lipinski_summary'],
            'BBB':            'Yes' if s['bbb_penetrates'] else 'No',
            'CNS MPO':        s['cns_mpo_score'],
            'CNS Class':      s['cns_class'],
            'Solubility':     s['solubility_class'],
            'Alerts':         s['num_alerts'],
            'Decision':       s['decision'],
        })
    else:
        print(f'  Error: {resp["error"]}')

df = pd.DataFrame(rows)
# Note: Atenolol vs Propranolol comparison is intentional —
# both are beta-blockers but Atenolol is BBB−, Propranolol is BBB+.
# This contrast motivates the optimization challenge in Notebook 2.
df

---
## 4. Define the Claude Tool Schemas

The Claude API expects tools as JSON schemas. Here are the three tools our agent will have:

| Tool | What it does |
|---|---|
| `validate_smiles` | Local RDKit check — fast, no API cost |
| `analyze_molecule` | Full ADMET scoring via local RDKit (instant) |
| `compare_candidates` | Side-by-side comparison of multiple molecules |

In [ ]:
TOOLS = [
    {
        "name": "validate_smiles",
        "description": (
            "Validate a SMILES string using RDKit (local, instant). "
            "Always call this FIRST before analyze_molecule to catch invalid chemistry early."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "smiles": {
                    "type": "string",
                    "description": "The SMILES string to validate"
                }
            },
            "required": ["smiles"]
        }
    },
    {
        "name": "analyze_molecule",
        "description": (
            "Analyze a molecule's full ADMET profile using local RDKit-based scoring. "
            "Returns QED score, Lipinski compliance, BBB penetration probability, CNS MPO score, "
            "solubility estimate, GI absorption, CYP450 flags, structural alerts, "
            "a triage decision (Progress/Optimize/Kill), and optimization suggestions. "
            "Instant — no network call required."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "smiles": {
                    "type": "string",
                    "description": "A valid SMILES string for the molecule to analyze"
                }
            },
            "required": ["smiles"]
        }
    },
    {
        "name": "compare_candidates",
        "description": (
            "Analyze and compare multiple molecules side by side using local RDKit scoring. "
            "Use this to evaluate all candidates after a round of optimization "
            "and identify which one best meets the target profile."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "smiles_list": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of SMILES strings to compare"
                },
                "labels": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Optional human-readable labels (e.g. ['Starting material', 'Round 1', 'Round 2'])"
                }
            },
            "required": ["smiles_list"]
        }
    }
]

print(f'{len(TOOLS)} tools defined:')
for t in TOOLS:
    print(f'  • {t["name"]}')
print('\nReady for Notebook 2 — the Agent Loop.')

---
## Summary

- We used the **local RDKit scorer** (`agent_utils.py`) to analyze real molecules — no network required
- We understand the full response shape — QED, BBB, CNS MPO, decision, suggestions
- We defined **3 Claude tool schemas** — the agent's action space

**Key insight:** The scorer already returns `decision_rationale` and `optimization_suggestions` from deterministic rules. The agent uses these as starting hints, then reasons about *how* to act on them — adding genuine chemical intelligence on top of fast local calculations.

➡️ **Next:** `02_agent_loop.ipynb` — the full agentic loop